# 03 — Documentação dos Treinamentos

Este notebook documenta os treinamentos realizados para os cenários C2, C3 e C4 do experimento de avaliação de defesas contra prompt injection.

Os cenários documentados são:

- C2: StruQ-like SFT;
- C3: SecAlign-like DPO;
- C4: Instruction-Hierarchy-like SFT.

O objetivo deste notebook é registrar:

- scripts de treinamento utilizados;
- comandos executados;
- seeds de treinamento;
- diretórios de saída dos adaptadores;
- logs de treinamento;
- métricas de custo computacional;
- evolução da loss;
- arquivos gerados por cada cenário.

## 0. Validação do kernel

Antes de executar qualquer célula Python deste notebook, é necessário garantir que o kernel selecionado seja o ambiente virtual do projeto:

```text
Python (pi-defense-exp)
```

Se o notebook estiver usando outro kernel, os scripts podem falhar por ausência de bibliotecas como `torch`, `transformers`, `peft`, `trl` ou `datasets`.

In [ ]:
import sys
from pathlib import Path

python_executable = Path(sys.executable)

print("Python executable:", python_executable)

expected_fragment = "/workspace/pi-defense-exp/.venv/bin/python"

if expected_fragment not in str(python_executable):
    raise RuntimeError(
        "Kernel incorreto. Selecione o kernel 'Python (pi-defense-exp)' antes de continuar."
    )

print("Kernel correto: Python (pi-defense-exp)")

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import re
import shutil
import subprocess
import sys

import pandas as pd
import numpy as np

print("Imports carregados com sucesso.")

In [ ]:
# Diretório principal do projeto
PROJECT_DIR = Path("/workspace/pi-defense-exp")
os.chdir(PROJECT_DIR)

# Python da venv
VENV_PYTHON = PROJECT_DIR / ".venv" / "bin" / "python"

# Diretórios principais
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

SCRIPTS_DIR = PROJECT_DIR / "scripts"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"

RUNS_DIR = PROJECT_DIR / "runs"

# Configuração do experimento
EXPERIMENT_NAME = "opi_sst2_sms_5k_872"
DATASET_SEED = 42
TRAINING_SEEDS = [42, 123, 2026]

RUN_ROOT = RUNS_DIR / EXPERIMENT_NAME / f"dataset_seed{DATASET_SEED}"
RUN_DATA_DIR = RUN_ROOT / "data"

# Flags de segurança
RUN_TRAINING = False

# Caso queira executar apenas parte do treino pelo notebook, ajuste estas listas.
SEEDS_TO_RUN = [42]
SCENARIOS_TO_RUN = ["C2", "C3", "C4"]

print("PROJECT_DIR:", PROJECT_DIR)
print("VENV_PYTHON:", VENV_PYTHON)
print("RUN_ROOT:", RUN_ROOT)
print("RUN_TRAINING:", RUN_TRAINING)

### 1 Autenticação no Hugging Face

O modelo base utilizado no experimento é um modelo gated do Hugging Face. Por isso, antes de executar treinamentos ou avaliações, o ambiente precisa estar autenticado com uma conta que tenha acesso ao repositório do modelo.

---
Alternativa pelo terminal

Se quiser resolver pelo terminal, faça login usando o mesmo HF_HOME do notebook:

```bash
cd /workspace/pi-defense-exp
source .venv/bin/activate

export HF_HOME=/workspace/pi-defense-exp/cache
export TRANSFORMERS_CACHE=/workspace/pi-defense-exp/cache/transformers
export HF_DATASETS_CACHE=/workspace/pi-defense-exp/cache/datasets

mkdir -p "$HF_HOME" "$TRANSFORMERS_CACHE" "$HF_DATASETS_CACHE"

hf auth login --force
hf auth whoami
```

Depois volte no notebook, reinicie o kernel e rode de novo.

In [ ]:
import os
from pathlib import Path
from getpass import getpass

PROJECT_DIR = Path("/workspace/pi-defense-exp")
CACHE_DIR = PROJECT_DIR / "cache"

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(CACHE_DIR / "transformers")
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "transformers").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "datasets").mkdir(parents=True, exist_ok=True)

hf_token = getpass("Cole seu token do Hugging Face: ")

from huggingface_hub import login

login(
    token=hf_token,
    add_to_git_credential=False,
)

print("Login realizado para o ambiente atual do notebook.")

In [ ]:
from huggingface_hub import whoami

user_info = whoami()
print("Autenticado como:", user_info.get("name"))

## 2. Cenários de treinamento

Os cenários treinados neste experimento são:

| Cenário | Nome | Tipo de treinamento | Arquivo de treino |
|---|---|---|---|
| C2 | StruQ-like SFT | Supervised Fine-Tuning com LoRA | `train_struq_sft.jsonl` |
| C3 | SecAlign-like DPO | Direct Preference Optimization com LoRA | `train_secalign_dpo.jsonl` |
| C4 | Instruction-Hierarchy-like SFT | Supervised Fine-Tuning com LoRA | `train_ih_sft.jsonl` |

Todos os treinamentos utilizam o mesmo modelo base e o mesmo dataset canônico. A diferença entre as execuções está no formato dos dados e na seed de treinamento.

In [ ]:
training_files = {
    "C2": PROCESSED_DATA_DIR / "train_struq_sft.jsonl",
    "C3": PROCESSED_DATA_DIR / "train_secalign_dpo.jsonl",
    "C4": PROCESSED_DATA_DIR / "train_ih_sft.jsonl",
}

training_file_checks = []

for scenario, path in training_files.items():
    training_file_checks.append({
        "scenario": scenario,
        "file": path.name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else None,
    })

training_file_checks_df = pd.DataFrame(training_file_checks)
training_file_checks_df

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0

    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


line_counts = []

for scenario, path in training_files.items():
    line_counts.append({
        "scenario": scenario,
        "file": path.name,
        "lines": count_jsonl_lines(path),
    })

pd.DataFrame(line_counts)

## 3. Scripts de treinamento

Dois scripts são utilizados para treinamento:

### `train_sft_lora_fp16.py`

Usado para os cenários:

- C2: StruQ-like SFT;
- C4: Instruction-Hierarchy-like SFT.

### `train_dpo_lora_fp16.py`

Usado para o cenário:

- C3: SecAlign-like DPO.

Ambos os scripts recebem o argumento `--seed`, permitindo repetir os treinamentos com seeds diferentes sem alterar o dataset.

In [ ]:
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print("SCRIPTS_DIR:", SCRIPTS_DIR)
print("Existe?", SCRIPTS_DIR.exists())

In [ ]:
%%writefile /workspace/pi-defense-exp/scripts/train_sft_lora_fp16.py
import os
import argparse
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed,
)
from peft import LoraConfig, get_peft_model, TaskType


def tokenize(example, tokenizer, max_length):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    result["labels"] = result["input_ids"].copy()
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_file", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--max_length", type=int, default=1024)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)

    print(f"[INFO] Seed de treino: {args.seed}")

    model_id = os.environ.get("MODEL_ID", "meta-llama/Meta-Llama-3-8B-Instruct")

    print(f"[INFO] Modelo base: {model_id}")
    print(f"[INFO] Arquivo de treino: {args.train_file}")
    print(f"[INFO] Diretório de saída: {args.output_dir}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    dataset = load_dataset("json", data_files=args.train_file, split="train")

    dataset = dataset.map(
        lambda x: tokenize(x, tokenizer, args.max_length),
        remove_columns=dataset.column_names,
    )

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        seed=args.seed,
        data_seed=args.seed,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",
        optim="adamw_torch",
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        warmup_steps=0,
    )

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    print(f"[OK] Adapter SFT salvo em: {args.output_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /workspace/pi-defense-exp/scripts/train_dpo_lora_fp16.py
import os
import argparse
import inspect
import torch
import trl

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType
from trl import DPOTrainer, DPOConfig


def filter_supported_kwargs(cls, kwargs):
    signature = inspect.signature(cls.__init__)
    valid_params = set(signature.parameters.keys())

    filtered = {}
    dropped = {}

    for key, value in kwargs.items():
        if key in valid_params:
            filtered[key] = value
        else:
            dropped[key] = value

    if dropped:
        print(f"[INFO] Argumentos ignorados por incompatibilidade com esta versão de {cls.__name__}:")
        for key in dropped:
            print(f"  - {key}")

    return filtered


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train_file", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--max_length", type=int, default=1024)
    parser.add_argument("--max_prompt_length", type=int, default=512)
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)

    print(f"[INFO] TRL version: {trl.__version__}")
    print(f"[INFO] Seed de treino: {args.seed}")

    model_id = os.environ.get("MODEL_ID", "meta-llama/Meta-Llama-3-8B-Instruct")

    print(f"[INFO] Modelo base: {model_id}")
    print(f"[INFO] Arquivo de treino: {args.train_file}")
    print(f"[INFO] Diretório de saída: {args.output_dir}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    )

    dataset = load_dataset("json", data_files=args.train_file, split="train")

    print("[INFO] Colunas do dataset:", dataset.column_names)
    print("[INFO] Exemplo 0:", dataset[0])

    dpo_config_kwargs = {
        "output_dir": args.output_dir,
        "seed": args.seed,
        "data_seed": args.seed,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": args.epochs,
        "learning_rate": 5e-5,
        "fp16": True,
        "logging_steps": 10,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "report_to": "none",
        "beta": 0.1,
        "max_length": args.max_length,
        "max_prompt_length": args.max_prompt_length,
        "gradient_checkpointing": True,
        "remove_unused_columns": False,
    }

    dpo_args = DPOConfig(
        **filter_supported_kwargs(DPOConfig, dpo_config_kwargs)
    )

    trainer_kwargs = {
        "model": model,
        "ref_model": None,
        "args": dpo_args,
        "train_dataset": dataset,
        "processing_class": tokenizer,
        "tokenizer": tokenizer,
        "peft_config": peft_config,
    }

    trainer = DPOTrainer(
        **filter_supported_kwargs(DPOTrainer, trainer_kwargs)
    )

    trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    print(f"[OK] Adapter DPO salvo em: {args.output_dir}")


if __name__ == "__main__":
    main()

In [ ]:
training_scripts = [
    SCRIPTS_DIR / "train_sft_lora_fp16.py",
    SCRIPTS_DIR / "train_dpo_lora_fp16.py",
]

compile_results = []

for script in training_scripts:
    result = subprocess.run(
        [str(VENV_PYTHON), "-m", "py_compile", str(script)],
        capture_output=True,
        text=True,
    )

    compile_results.append({
        "script": script.name,
        "returncode": result.returncode,
        "stderr": result.stderr.strip(),
        "path": str(script),
    })

compile_results_df = pd.DataFrame(compile_results)
compile_results_df

In [ ]:
if not (compile_results_df["returncode"] == 0).all():
    raise RuntimeError("Pelo menos um script de treinamento possui erro de sintaxe.")

print("Scripts de treinamento compilados com sucesso.")

## 4. Plano de treinamento

Os treinamentos são organizados por seed.

A estrutura versionada esperada é:

```text
runs/opi_sst2_sms_5k_872/dataset_seed42/train_seed<SEED>/
├── adapters
│   ├── c2_struq_sft
│   ├── c3_secalign_dpo
│   └── c4_instruction_hierarchy_sft
├── logs
├── raw
└── metrics
```

In [ ]:
def seed_root(seed: int) -> Path:
    return RUN_ROOT / f"train_seed{seed}"


def scenario_adapter_dir(seed: int, scenario: str) -> Path:
    mapping = {
        "C2": seed_root(seed) / "adapters" / "c2_struq_sft",
        "C3": seed_root(seed) / "adapters" / "c3_secalign_dpo",
        "C4": seed_root(seed) / "adapters" / "c4_instruction_hierarchy_sft",
    }
    return mapping[scenario]


def scenario_log_path(seed: int, scenario: str) -> Path:
    mapping = {
        "C2": seed_root(seed) / "logs" / "03_train_c2_struq_sft.log",
        "C3": seed_root(seed) / "logs" / "04_train_c3_secalign_dpo.log",
        "C4": seed_root(seed) / "logs" / "05_train_c4_instruction_hierarchy_sft.log",
    }
    return mapping[scenario]


def training_command(seed: int, scenario: str):
    if scenario == "C2":
        return [
            str(VENV_PYTHON),
            str(SCRIPTS_DIR / "train_sft_lora_fp16.py"),
            "--train_file", str(PROCESSED_DATA_DIR / "train_struq_sft.jsonl"),
            "--output_dir", str(scenario_adapter_dir(seed, scenario)),
            "--max_length", "1024",
            "--epochs", "1",
            "--seed", str(seed),
        ]

    if scenario == "C3":
        return [
            str(VENV_PYTHON),
            str(SCRIPTS_DIR / "train_dpo_lora_fp16.py"),
            "--train_file", str(PROCESSED_DATA_DIR / "train_secalign_dpo.jsonl"),
            "--output_dir", str(scenario_adapter_dir(seed, scenario)),
            "--max_length", "1024",
            "--max_prompt_length", "512",
            "--epochs", "1",
            "--seed", str(seed),
        ]

    if scenario == "C4":
        return [
            str(VENV_PYTHON),
            str(SCRIPTS_DIR / "train_sft_lora_fp16.py"),
            "--train_file", str(PROCESSED_DATA_DIR / "train_ih_sft.jsonl"),
            "--output_dir", str(scenario_adapter_dir(seed, scenario)),
            "--max_length", "1024",
            "--epochs", "1",
            "--seed", str(seed),
        ]

    raise ValueError(f"Cenário desconhecido: {scenario}")

In [ ]:
training_plan_rows = []

for seed in TRAINING_SEEDS:
    for scenario in ["C2", "C3", "C4"]:
        adapter_dir = scenario_adapter_dir(seed, scenario)
        log_path = scenario_log_path(seed, scenario)
        command = training_command(seed, scenario)

        adapter_dir.parent.mkdir(parents=True, exist_ok=True)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        training_plan_rows.append({
            "seed": seed,
            "scenario": scenario,
            "adapter_dir": str(adapter_dir),
            "log_path": str(log_path),
            "command": " ".join(command),
            "will_run": RUN_TRAINING and seed in SEEDS_TO_RUN and scenario in SCENARIOS_TO_RUN,
        })

training_plan_df = pd.DataFrame(training_plan_rows)
training_plan_df

### 4.1 Autenticação no Hugging Face

O modelo base utilizado no experimento é um modelo gated do Hugging Face. Por isso, antes de executar treinamentos ou avaliações, o ambiente precisa estar autenticado com uma conta que tenha acesso ao repositório do modelo.

A autenticação pode ser feita no terminal do RunPod com:

```bash
cd /workspace/pi-defense-exp
source .venv/bin/activate
hf auth login
```

Após inserir um token com permissão de leitura, é possível verificar a autenticação com:

```bash
hf auth whoami
```

## 5. Execução opcional dos treinamentos

A célula abaixo permite executar os treinamentos a partir do notebook.

Por padrão, `RUN_TRAINING=False`, para evitar reexecutar treinamentos acidentalmente.

Para executar treinamentos pelo notebook, ajuste:

```python
RUN_TRAINING = True
SEEDS_TO_RUN = [42]
SCENARIOS_TO_RUN = ["C2"]
```

In [ ]:
def run_command_streaming(command, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("Executando:")
    print(" ".join(command))
    print("Log:", log_path)
    print("=" * 100)

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=str(PROJECT_DIR),
        )

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)

        process.wait()

        if process.returncode != 0:
            raise RuntimeError(f"Comando falhou com returncode={process.returncode}")

    print("[OK] Execução concluída.")

In [ ]:
if RUN_TRAINING:
    for _, row in training_plan_df.iterrows():
        if not row["will_run"]:
            continue

        command = row["command"].split()
        log_path = Path(row["log_path"])
        run_command_streaming(command, log_path)
else:
    print("RUN_TRAINING=False. Nenhum treinamento será executado nesta célula.")
    print("Os comandos foram documentados em training_plan_df.")

## 6. Verificação dos adaptadores gerados

Após o treinamento, cada cenário deve gerar um diretório de adapter contendo arquivos como:

- `adapter_config.json`;
- `adapter_model.safetensors` ou arquivo equivalente;
- tokenizer;
- checkpoints, quando salvos durante o treinamento.

A célula abaixo verifica a existência e o tamanho dos adaptadores.

In [ ]:
adapter_rows = []

for seed in TRAINING_SEEDS:
    for scenario in ["C2", "C3", "C4"]:
        adapter_dir = scenario_adapter_dir(seed, scenario)
        adapter_model_files = list(adapter_dir.glob("adapter_model.*"))
        checkpoint_dirs = sorted(adapter_dir.glob("checkpoint-*"))

        adapter_rows.append({
            "seed": seed,
            "scenario": scenario,
            "adapter_dir": str(adapter_dir),
            "exists": adapter_dir.exists(),
            "has_adapter_config": (adapter_dir / "adapter_config.json").exists(),
            "has_adapter_model": len(adapter_model_files) > 0,
            "n_checkpoints": len(checkpoint_dirs),
            "size_mb": round(
                sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / 1024**2,
                3
            ) if adapter_dir.exists() else 0,
        })

adapter_check_df = pd.DataFrame(adapter_rows)
adapter_check_df

## 7. Extração dos logs de treinamento

Os logs de treinamento registram informações como:

- loss;
- grad_norm;
- learning_rate;
- epoch;
- train_runtime;
- train_samples_per_second;
- train_steps_per_second;
- train_loss.

A seguir, os arquivos `trainer_state.json` são usados para extrair as informações estruturadas quando disponíveis.

In [ ]:
def find_latest_trainer_state(adapter_dir: Path):
    trainer_states = list(adapter_dir.rglob("trainer_state.json"))

    if not trainer_states:
        return None

    def checkpoint_step(path: Path):
        match = re.search(r"checkpoint-(\d+)", str(path))
        return int(match.group(1)) if match else -1

    trainer_states = sorted(trainer_states, key=checkpoint_step)
    return trainer_states[-1]


def load_trainer_state(path: Path):
    if path is None or not path.exists():
        return None

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def extract_training_summary_from_state(state):
    if state is None:
        return {}

    log_history = state.get("log_history", [])

    final_train_log = {}
    loss_logs = []

    for row in log_history:
        if "loss" in row:
            loss_logs.append(row)
        if "train_runtime" in row or "train_loss" in row:
            final_train_log.update(row)

    summary = {
        "n_log_entries": len(log_history),
        "n_loss_entries": len(loss_logs),
        "final_train_loss": final_train_log.get("train_loss"),
        "train_runtime": final_train_log.get("train_runtime"),
        "train_samples_per_second": final_train_log.get("train_samples_per_second"),
        "train_steps_per_second": final_train_log.get("train_steps_per_second"),
        "final_epoch": final_train_log.get("epoch"),
        "global_step": state.get("global_step"),
        "best_metric": state.get("best_metric"),
    }

    return summary

In [ ]:
training_summary_rows = []
loss_curve_rows = []

for seed in TRAINING_SEEDS:
    for scenario in ["C2", "C3", "C4"]:
        adapter_dir = scenario_adapter_dir(seed, scenario)
        trainer_state_path = find_latest_trainer_state(adapter_dir)
        state = load_trainer_state(trainer_state_path)
        summary = extract_training_summary_from_state(state)

        row = {
            "seed": seed,
            "scenario": scenario,
            "adapter_dir": str(adapter_dir),
            "trainer_state_path": str(trainer_state_path) if trainer_state_path else None,
        }
        row.update(summary)
        training_summary_rows.append(row)

        if state:
            for log_row in state.get("log_history", []):
                if "loss" in log_row:
                    curve_row = {
                        "seed": seed,
                        "scenario": scenario,
                    }
                    curve_row.update(log_row)
                    loss_curve_rows.append(curve_row)

training_summary_df = pd.DataFrame(training_summary_rows)
loss_curve_df = pd.DataFrame(loss_curve_rows)

In [ ]:
training_summary_df.head(20)

In [ ]:
loss_curve_df.head(20)

## 8. Salvamento das tabelas consolidadas

As tabelas consolidadas de treinamento são salvas dentro da pasta da execução:

```text
runs/opi_sst2_sms_5k_872/dataset_seed42/
```

Esses arquivos facilitam a análise posterior sem precisar reprocessar todos os logs.

In [ ]:
training_summary_path = RUN_ROOT / "training_summary.csv"
loss_curve_path = RUN_ROOT / "training_loss_curve.csv"

training_summary_df.to_csv(training_summary_path, index=False)
loss_curve_df.to_csv(loss_curve_path, index=False)

print("Resumo dos treinamentos salvo em:", training_summary_path)
print("Curvas de loss salvas em:", loss_curve_path)

In [ ]:
pd.read_csv(training_summary_path)

## 9. Manifesto dos treinamentos

O manifesto registra os principais artefatos de treinamento associados a cada cenário e seed.

Ele inclui:

- seed de treinamento;
- cenário;
- diretório do adapter;
- caminho do log;
- existência do adapter;
- tamanho do adapter;
- caminho do `trainer_state.json`, quando disponível.

In [ ]:
training_manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset_seed": DATASET_SEED,
    "training_seeds": TRAINING_SEEDS,
    "created_at": datetime.now().isoformat(),
    "run_root": str(RUN_ROOT),
    "training_plan": training_plan_df.to_dict(orient="records"),
    "adapter_check": adapter_check_df.to_dict(orient="records"),
    "training_summary": training_summary_df.to_dict(orient="records"),
}

training_manifest_path = RUN_ROOT / "training_manifest.json"

with training_manifest_path.open("w", encoding="utf-8") as f:
    json.dump(training_manifest, f, ensure_ascii=False, indent=2)

print("Manifesto de treinamento salvo em:", training_manifest_path)

In [ ]:
with training_manifest_path.open("r", encoding="utf-8") as f:
    loaded_training_manifest = json.load(f)

loaded_training_manifest.keys()